Data Preparation and Cleaning Script

This script:
1. Loads both GeoPackage files 
2. Merges main file with construction year file via nhda_id
3. Filters to NHDA type only
4. Reshapes LST and NDVI columns from wide to long format
5. Creates analysis variables (years_since_construction_start, etc.)
6. Filters to max 9 years since construction start
7. Performs data quality checks
8. Saves prepared analysis dataset with CLEAR, EXPLICIT column names

Output:
    outputs/data_prepared/analysis_dataset.csv

In [27]:
import os
import sys
import warnings
from pathlib import Path
import re

import geopandas as gpd
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore', category=UserWarning)


# ============================================================================
# CONFIGURATION - MODIFY THESE PATHS
# ============================================================================

PATH_MAIN_GPKG = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Comparison_LST_NDVI_DEGURB_CLC_BuildingStructure_Differences_Census.gpkg"
PATH_CONSTRUCTION_GPKG = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\New_Housing_Development_Areas\NHDA_with_construction_years_RF_AUC.gpkg"

# Known column names
MERGE_ID_COL = 'nhda_id'  # Column to merge ond
CONSTRUCTION_YEAR_COL = 'construction_start_year'  # From construction file

# Output settings
OUTPUT_DIR = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Analysis_v2"
DATA_OUTPUT_DIR = os.path.join(OUTPUT_DIR)

# Analysis settings
MAX_YEARS_SINCE_CONSTRUCTION = 9  # Only analyze up to 9 years post-construction


# ============================================================================
# STEP 1: UTILITY FUNCTIONS (Inline - No External Dependencies)
# ============================================================================

def ensure_output_directory(base_path='outputs'):
    """Create output directory structure."""
    subdirs = {
        'base': base_path,
        'data': os.path.join(base_path, 'data_prepared'),
        'tables': os.path.join(base_path, 'tables'),
        'figures': os.path.join(base_path, 'figures'),
        'models': os.path.join(base_path, 'models'),
        'diagnostics': os.path.join(base_path, 'diagnostics'),
    }
    for path in subdirs.values():
        os.makedirs(path, exist_ok=True)
    return subdirs


def find_columns_by_pattern(gdf, patterns, verbose=False):
    """Find columns matching regex patterns."""
    if isinstance(patterns, str):
        patterns = [patterns]
    
    matching_cols = []
    for col in gdf.columns:
        for pattern in patterns:
            if re.search(pattern, col, re.IGNORECASE):
                matching_cols.append(col)
                break
    
    if verbose:
        print(f"Found columns matching {patterns}:")
        for col in matching_cols:
            print(f"  - {col}")
    
    return matching_cols


def extract_years_from_columns(columns):
    """Extract year from column names like 'difference_LST_2015'."""
    years = []
    for col in columns:
        match = re.search(r'_(\d{4})$', col)
        if match:
            years.append(int(match.group(1)))
    return sorted(set(years))


def print_section(title):
    """Print formatted section header."""
    print("\n" + "="*80)
    print(title)
    print("="*80 + "\n")


def print_columns(gdf, max_samples=3):
    """Print all column names with sample values."""
    print("\nAvailable columns:")
    for col in gdf.columns:
        if col == 'geometry':
            print(f"  ✓ {col:40s} (GEOMETRY - spatial data)")
        else:
            dtype = str(gdf[col].dtype)
            n_null = gdf[col].isna().sum()
            sample = str(gdf[col].dropna().iloc[0])[:30] if gdf[col].notna().any() else "N/A"
            print(f"  ✓ {col:40s} ({dtype:15s}) - null: {n_null:5d} - sample: {sample}")


# ============================================================================
# STEP 1: LOAD GEOPACKAGES (NO LAYER SELECTION - DIRECT LOAD)
# ============================================================================

print_section("STEP 1: LOADING GEOPACKAGES")

print(f"Loading main file: {PATH_MAIN_GPKG}")
try:
    gdf_main = gpd.read_file(PATH_MAIN_GPKG)
    print(f"✓ Loaded main file: {len(gdf_main)} rows × {gdf_main.shape[1]} columns")
except Exception as e:
    print(f"✗ ERROR loading main file: {e}")
    sys.exit(1)


print(f"\nLoading construction file: {PATH_CONSTRUCTION_GPKG}")
try:
    gdf_construction = gpd.read_file(PATH_CONSTRUCTION_GPKG)
    print(f"✓ Loaded construction file: {len(gdf_construction)} rows × {gdf_construction.shape[1]} columns")
except Exception as e:
    print(f"✗ ERROR loading construction file: {e}")
    sys.exit(1)


# ============================================================================
# STEP 2: INSPECT COLUMNS
# ============================================================================

print_section("STEP 2: INSPECTING COLUMNS IN MAIN FILE")

print("Main file columns:")
print_columns(gdf_main)

print_section("STEP 3: INSPECTING COLUMNS IN CONSTRUCTION FILE")

print("Construction file columns:")
print_columns(gdf_construction)


# ============================================================================
# STEP 4: IDENTIFY AND VERIFY KEY COLUMNS
# ============================================================================

print_section("STEP 4: IDENTIFYING KEY COLUMNS")

# Check for merge ID
if MERGE_ID_COL not in gdf_main.columns:
    print(f"✗ ERROR: '{MERGE_ID_COL}' not found in main file!")
    print(f"Available ID columns: {[c for c in gdf_main.columns if 'id' in c.lower()]}")
    sys.exit(1)
else:
    print(f"✓ Found merge ID in main file: '{MERGE_ID_COL}'")

if MERGE_ID_COL not in gdf_construction.columns:
    print(f"✗ ERROR: '{MERGE_ID_COL}' not found in construction file!")
    print(f"Available ID columns: {[c for c in gdf_construction.columns if 'id' in c.lower()]}")
    sys.exit(1)
else:
    print(f"✓ Found merge ID in construction file: '{MERGE_ID_COL}'")

# Check for construction year
if CONSTRUCTION_YEAR_COL not in gdf_construction.columns:
    print(f"✗ ERROR: '{CONSTRUCTION_YEAR_COL}' not found in construction file!")
    possible_year_cols = [c for c in gdf_construction.columns if 'year' in c.lower()]
    print(f"Possible year columns: {possible_year_cols}")
    sys.exit(1)
else:
    print(f"✓ Found construction year column: '{CONSTRUCTION_YEAR_COL}'")

# Find LST and NDVI difference columns
lst_cols = find_columns_by_pattern(gdf_main, r'difference_LST|LST_diff')
ndvi_cols = find_columns_by_pattern(gdf_main, r'difference_NDVI|NDVI_diff')

print(f"\n✓ Found {len(lst_cols)} LST difference columns: {lst_cols}")
print(f"✓ Found {len(ndvi_cols)} NDVI difference columns: {ndvi_cols}")

if len(lst_cols) == 0 or len(ndvi_cols) == 0:
    print("✗ ERROR: No LST or NDVI difference columns found!")
    sys.exit(1)

# Extract years
years = extract_years_from_columns(lst_cols + ndvi_cols)
print(f"✓ Available years: {years}")


# ============================================================================
# STEP 5: FILTER TO NHDA TYPE ONLY
# ============================================================================

print_section("STEP 5: FILTERING TO NHDA TYPE ONLY")

# Check for type column
if 'type' not in gdf_main.columns:
    print("✗ ERROR: 'type' column not found!")
    type_cols = [c for c in gdf_main.columns if 'type' in c.lower()]
    print(f"Possible type columns: {type_cols}")
    sys.exit(1)

print(f"Total rows before filtering: {len(gdf_main)}")
print(f"Unique types: {gdf_main['type'].unique()}")

# Filter to NHDA
gdf_main = gdf_main[gdf_main['type'] == 'NHDA'].copy()
print(f"✓ Filtered to NHDA type: {len(gdf_main)} rows remaining")

# Calculate centroid coordinates (WGS84)
if gdf_main.crs is not None and gdf_main.crs.to_epsg() != 4326:
    gdf_coords = gdf_main.to_crs(epsg=4326)
else:
    gdf_coords = gdf_main.copy()

centroids = gdf_coords.geometry.centroid

gdf_main["lon"] = centroids.x
gdf_main["lat"] = centroids.y

print(f"✓ Calculated centroid coordinates for {len(gdf_main)} NHDAs")

# ============================================================================
# STEP 6: MERGE WITH CONSTRUCTION YEAR
# ============================================================================

print_section("STEP 6: MERGING WITH CONSTRUCTION YEAR")

# Convert to regular DataFrame (no geometry needed for analysis)
df_main = pd.DataFrame(gdf_main.drop(columns=['geometry']))
df_construction = pd.DataFrame(gdf_construction.drop(columns=['geometry']))

print(f"Main file: {len(df_main)} rows")
print(f"Construction file: {len(df_construction)} rows")

# Keep only ID and construction year from construction file
df_construction = df_construction[[MERGE_ID_COL, CONSTRUCTION_YEAR_COL]].copy()
df_construction = df_construction.drop_duplicates(subset=[MERGE_ID_COL])
print(f"Construction file (deduplicated): {len(df_construction)} unique IDs")

# Merge
df = df_main.merge(df_construction, on=MERGE_ID_COL, how='left')
n_with_year = df[CONSTRUCTION_YEAR_COL].notna().sum()
print(f"✓ Merged: {n_with_year}/{len(df)} rows have construction year")

# Handle special values in construction_start_year
df[CONSTRUCTION_YEAR_COL] = (
    df[CONSTRUCTION_YEAR_COL]
    .replace({
        'AUC_2015': 2014,
        'AUC_2016': 2015
    })
)

# Convert everything to numeric
df[CONSTRUCTION_YEAR_COL] = pd.to_numeric(
    df[CONSTRUCTION_YEAR_COL],
    errors='coerce'
).astype('Int64')

if n_with_year == 0:
    print("✗ ERROR: No rows matched during merge! Check MERGE_ID_COL values")
    sys.exit(1)

# ============================================================================
# STEP 7: RESHAPE OUTCOMES TO LONG FORMAT
# ============================================================================

print_section("STEP 7: RESHAPING OUTCOMES TO LONG FORMAT")

# Prepare for reshape
id_col = MERGE_ID_COL

# Create long format
dfs_long = []

for year in years:
    # LST
    lst_col = None
    for col in lst_cols:
        if col.endswith(f'_{year}'):
            lst_col = col
            break
    
    # NDVI
    ndvi_col = None
    for col in ndvi_cols:
        if col.endswith(f'_{year}'):
            ndvi_col = col
            break
    
    if lst_col and ndvi_col:
        subset = df[[id_col, lst_col, ndvi_col, CONSTRUCTION_YEAR_COL]].copy()
        subset['observation_year'] = year
        subset = subset.rename(columns={
            lst_col: 'difference_LST',
            ndvi_col: 'NDVI_difference'
        })
        dfs_long.append(subset)

df_long = pd.concat(dfs_long, ignore_index=True)
print(f"✓ Reshaped to long format: {len(df_long)} rows (one per NHDA-year)")

# Re-merge with original data for predictors
df_main_keep = df[[
    id_col, CONSTRUCTION_YEAR_COL, "lon", "lat",

    # residential subclass shares
    'diff_res_subclass_share_mfh_ab',
    'diff_res_subclass_share_sbd',
    'diff_res_subclass_share_tb',
    'diff_res_subclass_share_sfh_db',

    # morphology predictors
    'reldiff_building_volume_density',
    'reldiff_building_density',
    'reldiff_avg_building_footprint',
    'reldiff_avg_building_height',
    'reldiff_built_up_ratio',

    # confounder
    'nhda_degurba_code'
]].copy()

# Add CLC columns (multiple possible names)
clc_cols = find_columns_by_pattern(df, r'nhda.*clc|nhda.*sur_class')
if clc_cols:
    clc_col = clc_cols[0]
    print(f"\n✓ Found CLC column: {clc_col}")
    df_main_keep[clc_col] = df[clc_col]

df_main_keep = df_main_keep.drop_duplicates(subset=[id_col])

# Merge back
df_final = df_long.merge(df_main_keep, on=[id_col, CONSTRUCTION_YEAR_COL], how='left')
print(f"✓ Merged with predictors: {len(df_final)} rows")


# ============================================================================
# STEP 8: CREATE ANALYSIS VARIABLES
# ============================================================================

print_section("STEP 8: CREATING ANALYSIS VARIABLES")

# Calculate years since construction
df_final['years_since_construction_start'] = df_final['observation_year'] - df_final[CONSTRUCTION_YEAR_COL]
print(f"✓ Created years_since_construction_start")
print(f"  Range: {df_final['years_since_construction_start'].min()} to {df_final['years_since_construction_start'].max()}")

# Create pre-construction baseline as average BEFORE construction
# ============================================================================
# Calculate mean pre- and post-construction differences per NHDA
# ============================================================================

# Pre-construction period
pre_values = (
    df_final[df_final["years_since_construction_start"] < 0]
    .groupby(id_col)
    .agg(
        pre_construction_difference_LST=(
            "difference_LST",
            "mean"
        ),
        pre_construction_difference_NDVI=(
            "NDVI_difference",
            "mean"
        )
    )
    .reset_index()
)

# Post-construction period
post_values = (
    df_final[df_final["years_since_construction_start"] >= 0]
    .groupby(id_col)
    .agg(
        post_construction_difference_LST=(
            "difference_LST",
            "mean"
        ),
        post_construction_difference_NDVI=(
            "NDVI_difference",
            "mean"
        )
    )
    .reset_index()
)

# Combine pre- and post-construction values
baseline = pre_values.merge(
    post_values,
    on=id_col,
    how="outer"
)

print(baseline.head())

# Filter to years 1 to 9 after construction start
n_before = len(df_final)
df_final = df_final[
    (df_final['years_since_construction_start'] >= 1) & 
    (df_final['years_since_construction_start'] <= MAX_YEARS_SINCE_CONSTRUCTION)
].copy()
n_after = len(df_final)

print(
    f"✓ Filtered to years 1-{MAX_YEARS_SINCE_CONSTRUCTION} "
    f"after construction start: {n_before} → {n_after} rows"
)

df_final = df_final.merge(baseline, on=id_col, how='left')
print("\n✓ Created average pre-construction LST/NDVI differences")

# Handle CLC column
clc_cols_available = find_columns_by_pattern(df_final, r'nhda.*clc|nhda.*sur_class')
if clc_cols_available:
    clc_col = clc_cols_available[0]
    print(f"\n✓ Processing CLC column: '{clc_col}'")
    
    # Create numeric version if needed
    df_final['nhda_sur_class_2021'] = pd.to_numeric(df_final[clc_col], errors='coerce').astype('Int64')
    
    # Create short version (first digit)
    df_final['nhda_sur_class_2021_short'] = (
        df_final['nhda_sur_class_2021'].astype(str).str[0].astype('Int64')
    )
    print(f"  - nhda_sur_class_2021: numeric version")
    print(f"  - nhda_sur_class_2021_short: first digit only (1-5)")
    print(f"  Values: {sorted(df_final['nhda_sur_class_2021_short'].dropna().unique())}")

# ============================================================================
# STEP 9: DATA QUALITY CHECKS
# ============================================================================

print_section("STEP 9: DATA QUALITY CHECKS")

print("Missing data summary:")
missing_data = df_final.isnull().sum()
missing_pct = (df_final.isnull().sum() / len(df_final) * 100).round(1)
for col in missing_data[missing_data > 0].index:
    print(f"  {col:40s}: {missing_data[col]:5d} rows ({missing_pct[col]:5.1f}%)")

print("\nSample size by years_since_construction_start:")
sample_by_year = df_final.groupby('years_since_construction_start').size()
for year in sorted(sample_by_year.index):
    n = sample_by_year[year]
    pct = n / len(df_final) * 100
    print(f"  Year {int(year):2d}: {n:5d} rows ({pct:5.1f}%)")

print("\nOutcome variable summaries:")
for outcome in ['difference_LST', 'NDVI_difference']:
    if outcome in df_final.columns:
        n_valid = df_final[outcome].notna().sum()
        print(f"\n  {outcome}:")
        print(f"    N valid: {n_valid}")
        print(f"    Mean: {df_final[outcome].mean():.4f}")
        print(f"    Std: {df_final[outcome].std():.4f}")
        print(f"    Range: [{df_final[outcome].min():.4f}, {df_final[outcome].max():.4f}]")

print("\nCategorical variables:")
for col in ['nhda_degurba_code', 'nhda_sur_class_2021_short']:
    if col in df_final.columns:
        print(f"  {col}: {sorted(df_final[col].dropna().unique())}")

df_final = df_final.rename(columns={
    "NDVI_difference": "difference_NDVI"
})

# # ============================================================================
# # STEP 10: SELECT AND SAVE FINAL COLUMNS
# # ============================================================================

# print_section("STEP 10: FINALIZING AND SAVING DATA")

# # Define final column order with EXPLICIT names
final_cols = [
    id_col,
    'observation_year',
    CONSTRUCTION_YEAR_COL,
    'years_since_construction_start',

     # centroid coordinates
    'lon',
    'lat',

    # outcomes
    'difference_LST',
    'difference_NDVI',

    # subclass share differences
    'diff_res_subclass_share_mfh_ab',
    'diff_res_subclass_share_sbd',
    'diff_res_subclass_share_tb',
    'diff_res_subclass_share_sfh_db',

    # morphology predictors
    'reldiff_building_volume_density',
    'reldiff_building_density',
    'reldiff_avg_building_footprint',
    'reldiff_avg_building_height',
    'reldiff_built_up_ratio',

    # confounders
    'nhda_degurba_code',
    'nhda_sur_class_2021',
    'nhda_sur_class_2021_short',
    'pre_construction_difference_LST',
    'pre_construction_difference_NDVI',
    'post_construction_difference_LST',
    'post_construction_difference_NDVI'
]

# Keep only columns that exist
final_cols = [c for c in final_cols if c in df_final.columns]

df_export = df_final[final_cols].copy()

print(f"Final dataset shape: {df_export.shape[0]} rows × {df_export.shape[1]} columns")
print("\nFinal column names (EXPLICIT AND CLEAR):")
for i, col in enumerate(final_cols, 1):
    print(f"  {i:2d}. {col}")

# Create output directory
os.makedirs(DATA_OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(DATA_OUTPUT_DIR, 'analysis_dataset.csv')

# Save
df_export.to_csv(output_path, index=False)
print(f"\n✓ Saved to: {output_path}")

# Show sample
print("\nFirst 5 rows of final dataset:")
print(df_export.head())

print("\nData types:")
print(df_export.dtypes)

# ============================================================================
# COMPLETION
# ============================================================================

print("\n" + "="*80)
print("✓ DATA PREPARATION COMPLETE")
print("="*80)
print(f"\nOutput file: {output_path}")



STEP 1: LOADING GEOPACKAGES

Loading main file: C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Comparison_LST_NDVI_DEGURB_CLC_BuildingStructure_Differences_Census.gpkg
✓ Loaded main file: 1678 rows × 200 columns

Loading construction file: C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\New_Housing_Development_Areas\NHDA_with_construction_years_RF_AUC.gpkg
✓ Loaded construction file: 839 rows × 44 columns

STEP 2: INSPECTING COLUMNS IN MAIN FILE

Main file columns:

Available columns:
  ✓ nhda_id                                  (str            ) - null:     0 - sample: 09161_1
  ✓ type                                     (str            ) - null:     0 - sample: NHDA
  ✓ area_ha                                  (float64        ) - null:     0 - sample: 1.149258764916533
  ✓ nhda_median_NDVI_2015                    (float64        ) - null:   958 - sample: 0.8899472045898438
  ✓ nhda_median_NDVI_2016                    (float64       